# HSI playground

Ad-hoc scratchpad wired to the same functions the pipeline uses (`hsi_workflow`).
Use it to selectively look at bands/layers/masks/spectra and to try preprocessing
or analysis settings before committing them to `config.py`.

For slider-driven tuning use the standalone tools instead:
`python debug_preprocess.py --dataset ...` and `python debug_masks.py --dataset ...`
(both accept `--demo` if no data is on disk).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # repo root, so hsi_workflow imports

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from hsi_workflow.config import DATASETS, WorkflowConfig, PieceConfig, RoiConfig
from hsi_workflow.cube_io import load_dataset_cube, iter_cube_paths
from hsi_workflow.preprocessing import (preprocess, savgol_smooth,
                                        normalize_intensity, noise_metrics)
from hsi_workflow.pieces import extract_pieces, foreground_distance
from hsi_workflow.rois import tile_rois
from hsi_workflow.decomposition import fit_pca
from hsi_workflow.clustering import cluster, cluster_map
from hsi_workflow.anomaly import fit_detectors, anomaly_map, to_probability
from hsi_workflow.viz import pseudo_rgb

DATASET = "SIO2_DISH_WHITE_20"          # <-- pick a preset (see DATASETS)
CUBE_INDEX = 0                    # which scan of the dataset
CROP = None                       # e.g. (100, 500, 100, 500) for big scans

ds = DATASETS[DATASET.lower()]
label, hdr = iter_cube_paths(ds)[CUBE_INDEX]
cube = load_dataset_cube(hdr, ds)
raw = cube.data if CROP is None else cube.data[CROP[0]:CROP[1], CROP[2]:CROP[3], :]
wl = cube.wavelengths
print(label, raw.shape, f"{wl[0]:.0f}-{wl[-1]:.0f} nm")

sio2 all 20 dish white (1417, 900, 300) 368-1008 nm


## 1. Preprocessing variants side by side
Pick a pixel and compare raw / calibrated / smoothed / SNV.

In [2]:
from hsi_workflow.cube_io import load_reference_spectrum
from hsi_workflow.preprocessing import calibrate_reflectance

white, sw = load_reference_spectrum(ds.white_ref)
dark, sd = load_reference_spectrum(ds.dark_ref)
refl = calibrate_reflectance(raw, cube.shutter, white, sw, dark, sd)

SG_WINDOW, SG_POLY = 11, 2        # <-- tune these
smooth = savgol_smooth(refl, SG_WINDOW, SG_POLY)
snv = normalize_intensity(smooth, "snv")

r, c = raw.shape[0] // 2, raw.shape[1] // 2   # <-- pick a pixel
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(wl, refl[r, c]);   axes[0].set_title("calibrated reflectance")
axes[1].plot(wl, refl[r, c], "0.7", lw=0.8); axes[1].plot(wl, smooth[r, c], "r")
axes[1].set_title(f"+ Savitzky-Golay (w={SG_WINDOW}, p={SG_POLY})")
axes[2].plot(wl, snv[r, c]);    axes[2].set_title("+ SNV")
for ax in axes: ax.set_xlabel("wavelength (nm)")
plt.tight_layout()

print("noise before:", noise_metrics(refl, SG_WINDOW, SG_POLY, sample=2000))
print("noise after :", noise_metrics(smooth, SG_WINDOW, SG_POLY, sample=2000))

noise before: {'rms_noise': 0.004332519822660706, 'snr': 61.34989172193675, 'n_pixels': 2000}
noise after : {'rms_noise': 0.0007074327053823436, 'snr': 375.72431668636875, 'n_pixels': 2000}


## 2. Band browsing
Look at individual wavelengths -- which bands reveal structure?

In [ ]:
targets = [400, 550, 700, 850, 1000]          # <-- nm to inspect
fig, axes = plt.subplots(1, len(targets), figsize=(4 * len(targets), 4))
for ax, t in zip(axes, targets):
    b = int(np.argmin(np.abs(wl - t)))
    im = ax.imshow(smooth[:, :, b], cmap="gray")
    ax.set_title(f"{wl[b]:.0f} nm"); ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Piece mask
Tune extraction, view the distance map + mask overlay.

In [ ]:
pc = PieceConfig(method="sam", open_iter=2, close_iter=6, min_area=1000)  # <-- tune
dist = foreground_distance(raw, pc)
pieces = extract_pieces(cube, pc)
print(f"{len(pieces)} piece(s):", [(p.piece_id, int(p.mask.sum())) for p in pieces])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].imshow(pseudo_rgb(raw, wl)); axes[0].set_title("pseudo-RGB")
axes[1].imshow(dist, cmap="magma"); axes[1].set_title(f"foreground distance ({pc.method})")
b = int(np.argmin(np.abs(wl - 700)))
axes[2].imshow(raw[:, :, b], cmap="gray")
full_mask = np.zeros(raw.shape[:2], bool)
for p in pieces:
    r0, r1, c0, c1 = p.bbox
    full_mask[r0:r1, c0:c1] |= p.mask
axes[2].imshow(np.ma.masked_where(~full_mask, full_mask), cmap="autumn", alpha=0.4)
axes[2].set_title("mask overlay")
for ax in axes: ax.axis("off")
plt.tight_layout()

## 4. ROI tiling
How many patches survive per piece at these settings?

In [ ]:
rc = RoiConfig(patch=32, stride=32, min_coverage=0.85)     # <-- tune
for p in pieces:
    rois = tile_rois(p, rc)
    print(f"{p.piece_id}: {len(rois)} ROI(s)")

## 5. Quick PCA / clusters / anomaly on one piece

In [ ]:
wf = WorkflowConfig()
piece = pieces[2]
# analysis spectra for this piece (SNV of the smoothed reflectance crop)
r0, r1, c0, c1 = piece.bbox
adata = normalize_intensity(savgol_smooth(refl[r0:r1, c0:c1, :], SG_WINDOW, SG_POLY), "snv")
fg = adata[piece.mask]

pca = fit_pca(fg, wf.pca)
feat = pca.transform(fg)
print("explained variance:", np.round(pca.explained_variance_ratio, 3))

cres = cluster(feat, wf.cluster)
cmap_img = cluster_map(cres, piece.mask.shape, piece.mask)

det = fit_detectors(feat, wf.anomaly)[wf.anomaly.methods[0]]
amap = anomaly_map(det.score(feat), piece.mask.shape, piece.mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(feat[::5, 0], feat[::5, 1], s=2, alpha=0.3)
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2"); axes[0].set_title("PCA scatter")
axes[1].imshow(np.ma.masked_where(cmap_img < 0, cmap_img), cmap="tab10", interpolation="nearest")
axes[1].set_title("clusters"); axes[1].axis("off")
axes[2].imshow(np.ma.masked_invalid(to_probability(amap)), cmap="inferno")
axes[2].set_title("anomaly probability (0-1)"); axes[2].axis("off")
plt.tight_layout()

In [ ]:
import spectral.io.envi as envi
img = envi.open(r"C:\Users\shash\OneDrive - purdue.edu\Summer\hsi\sio2\sio2 all 20 dish white.bil.hdr")
cube = img.load()

In [ ]:
import spectral as spy
rgb_img = spy.imshow(img, bands=(85,42, 15))
plt.show()

In [ ]:
swir2 = cube[:, :, 90].astype(float)
swir1 = cube[:, :, 70].astype(float)

# Calculate ratio (adding epsilon to avoid division by zero)
silicon_index = swir2 / (swir1 + 1e-6)

plt.imshow(silicon_index, cmap='viridis')
plt.colorbar(label='Silicon Index Value')
plt.title('Silicon / Silicate Relative Concentration Map')
plt.show()

In [ ]:
# Select band indices corresponding to maximum interference contrast vs baseline
# e.g., Band 15 (Interference minimum) and Band 60 (Baseline)
band_trough = cube[:, :, 15].astype(float)
band_baseline = cube[:, :, 60].astype(float)

# Compute normalized ratio
ratio_map = (band_trough - band_baseline) / (band_trough + band_baseline + 1e-6)

# Visualize contrast map
plt.imshow(ratio_map, cmap='coolwarm')
plt.colorbar(label='Interference Ratio')
plt.title('SiO2 vs Bare Silicon Contrast')
plt.show()

In [ ]:
bare_si_cube = spy.open_image(r"C:\Users\shash\OneDrive - purdue.edu\Summer\hsi\sio2\bare silicon square.bip.hdr").load()
spy.imshow(bare_si_cube, bands=(85,42, 15))
plt.show()

In [ ]:
sample_cube = cube
bare_si_cube = spy.open_image(r"C:\Users\shash\OneDrive - purdue.edu\Summer\hsi\sio2\bare silicon large square.bip.hdr").load()

bare_si_ref = np.mean(bare_si_cube, axis=(0, 1))
bare_si_ref = np.atleast_2d(bare_si_ref) # or [bare_si_ref_1d]
angles = spy.spectral_angles(sample_cube, bare_si_ref)  # Output shape: (Rows, Cols)

# 3. Threshold: Angle close to 0 = Bare Silicon, Higher Angle = SiO2 coatedlarge
# (Adjust threshold value based on your angle distribution)
threshold = 0.1  # radians
silicon_mask = angles < threshold
sio2_mask = angles >= threshold

# 4. Display binary separation map
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(silicon_mask, cmap='gray')
ax[0].set_title('Bare Silicon Region')

ax[1].imshow(sio2_mask, cmap='gray')
ax[1].set_title('SiO2 Layer Region')
plt.show()

In [ ]:
import spectral as spy

# Compute SAM angles against the pure Si reference spectrum
# Returns a matrix of angles (in radians)
sam_angles = spy.spectral_angles(sample_cube, bare_si_cube)

# Threshold to separate Si and SiO2
sio2_mask = sam_angles > 0.1  # Adjust threshold based on your signal contrast

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(sam_angles, cmap='jet')
plt.title('Spectral Angle Map (Radiant difference from Si)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(sio2_mask, cmap='gray')
plt.title('SiO2 Binary Mask')
plt.show()

## Next
- Freeze what worked into `hsi_workflow/config.py` (the debug tools print
  paste-ready config snippets with `p`).
- Full runs: `run_organize` -> `run_explore` -> `run_analyze` (see `docs/usage.md`).